## Camada Gold - Modelos Analíticos
Criação da tabela agregada `fato_logistica` para responder diretamente às perguntas de negócio.

In [0]:
from pyspark.sql import functions as F

### Uso do schema

In [0]:
catalog = "mvp_data_engeneering"
schema_origem = "silver"
schema = "gold"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

### Leitura das tabelas da camada Silver

In [0]:

df_orders = spark.table(f"{catalog}.{schema_origem}.orders_cleaned")
df_reviews = spark.table(f"{catalog}.{schema_origem}.reviews_cleaned")
df_items = spark.table(f"{catalog}.{schema_origem}.order_sellers")
df_customers = spark.table(f"{catalog}.{schema_origem}.customers_cleaned")
df_sellers = spark.table(f"{catalog}.{schema_origem}.sellers_cleaned")

### Criação do modelo denormalizado 

In [0]:

df_gold = df_orders \
    .join(df_items, on="order_id", how="left") \
    .join(df_reviews, on="order_id", how="left") \
    .join(df_customers, on="customer_id", how="left") \
    .join(df_sellers, on="seller_id", how="left") \
    .select(
        "order_id",
        "customer_id",
        "seller_id",
        F.col("customer_state").alias("uf_cliente"),
        F.col("seller_state").alias("uf_vendedor"),
        "dias_processamento",
        "dias_transporte",
        "dias_atraso",
        "is_atrasado",
        "review_score"
    )


### Guardar a tabela na camada Gold

In [0]:

df_gold.write.format("delta").mode("overwrite").saveAsTable("fato_logistica")
print("Tabela fato_logistica gerada e guardada com sucesso na Gold.")